In [1]:
# autogluon smoketest

In [ ]:
# AutoGluon MultiModal Image Classification — SMOKE TEST (portal-friendly)
# Generates synthetic images -> builds DataFrames -> trains -> evaluates -> predicts

import random, shutil, time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

import torch

# --- Must work (setuptools <82) ---
import setuptools, pkg_resources
print("setuptools:", setuptools.__version__)
print("pkg_resources OK")

from autogluon.multimodal import MultiModalPredictor

# -------------------------
# 0) Environment check
# -------------------------
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -------------------------
# 1) Generate synthetic images (3 easy classes)
# -------------------------
CLASSES = ["red_square", "green_circle", "blue_triangle"]
IMG_SIZE = 64

# keep small so it runs fast on portals
N_TRAIN_PER_CLASS = 40
N_VAL_PER_CLASS   = 10
N_TEST_PER_CLASS  = 10

root = Path("./ag_smoketest_data")
train_dir = root / "train"
val_dir   = root / "val"
test_dir  = root / "test"

if root.exists():
    shutil.rmtree(root)
for d in [train_dir, val_dir, test_dir]:
    d.mkdir(parents=True, exist_ok=True)

def draw_one(label: str, rng: np.random.Generator) -> Image.Image:
    bg = rng.integers(220, 256, size=(IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    img = Image.fromarray(bg, mode="RGB")
    d = ImageDraw.Draw(img)

    margin = int(rng.integers(6, 14))
    x0 = int(rng.integers(margin, 20)); y0 = int(rng.integers(margin, 20))
    x1 = int(rng.integers(44, IMG_SIZE - margin)); y1 = int(rng.integers(44, IMG_SIZE - margin))

    if label == "red_square":
        d.rectangle([x0, y0, x1, y1], fill=(230, 30, 30))
    elif label == "green_circle":
        d.ellipse([x0, y0, x1, y1], fill=(30, 200, 60))
    elif label == "blue_triangle":
        p1 = (int(rng.integers(margin, 20)), int(rng.integers(40, IMG_SIZE - margin)))
        p2 = (int(rng.integers(24, IMG_SIZE - margin)), int(rng.integers(margin, 20)))
        p3 = (int(rng.integers(40, IMG_SIZE - margin)), int(rng.integers(40, IMG_SIZE - margin)))
        d.polygon([p1, p2, p3], fill=(40, 80, 230))
    else:
        raise ValueError("unknown label")

    # tiny pixel noise
    arr = np.array(img)
    arr = np.clip(arr + rng.integers(0, 12, size=arr.shape, dtype=np.uint8), 0, 255).astype(np.uint8)
    return Image.fromarray(arr, mode="RGB")

def make_split(out_dir: Path, n_per_class: int, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    rows = []
    for label in CLASSES:
        for i in range(n_per_class):
            fp = out_dir / f"{label}_{i:04d}.png"
            draw_one(label, rng).save(fp)
            rows.append({"image": str(fp), "label": label})
    return pd.DataFrame(rows)

train_df = make_split(train_dir, N_TRAIN_PER_CLASS, seed=SEED + 1)
val_df   = make_split(val_dir,   N_VAL_PER_CLASS,   seed=SEED + 2)
test_df  = make_split(test_dir,  N_TEST_PER_CLASS,  seed=SEED + 3)  # labels only for sanity-check

print("train/val/test shapes:", train_df.shape, val_df.shape, test_df.shape)
print(train_df.head(3))

# -------------------------
# 2) Train MultiModalPredictor
# -------------------------
save_path = "./ag_mm_smoketest_model"
if Path(save_path).exists():
    shutil.rmtree(save_path)

predictor = MultiModalPredictor(label="label", path=save_path, problem_type="multiclass")

t0 = time.time()
predictor.fit(
    train_data=train_df,
    tuning_data=val_df,
    time_limit=120,            # bump to 300 if portal is slow
    presets="medium_quality",  # valid AutoMM preset
)
print(f"fit done in {time.time() - t0:.1f}s")

# -------------------------
# 3) Evaluate + predict
# -------------------------
print("val metrics:", predictor.evaluate(val_df))

test_pred = predictor.predict(test_df.drop(columns=["label"]))
test_acc = (test_pred.values == test_df["label"].values).mean()
print("test acc (synthetic sanity):", round(float(test_acc), 4))

print(pd.DataFrame({
    "image": test_df["image"].head(10),
    "true":  test_df["label"].head(10),
    "pred":  test_pred.head(10),
}))


In [ ]:
import sys
import torch
import setuptools
import autogluon.core as ag
from autogluon.multimodal import MultiModalPredictor

def check_autogluon_multimodal():
    print("--- Environment Check for AutoGluon Multimodal ---")
    
    # 1. Check Python and Package Versions
    print(f"Python Version: {sys.version.split()[0]}")
    print(f"AutoGluon Version: {ag.__version__}")
    
    # 2. Check PyTorch & GPU Availability
    # Image classification in AutoMM heavily relies on PyTorch
    cuda_available = torch.cuda.is_available()
    print(f"PyTorch Version: {torch.__version__}")
    print(f"GPU Available (CUDA): {cuda_available}")
    
    if cuda_available:
        print(f"GPU Device: {torch.cuda.get_device_name(0)}")
        print(f"CUDA Version: {torch.version.cuda}")
    else:
        print("WARNING: No GPU detected. Image classification will be extremely slow on CPU.")

    # 3. Test MultiModalPredictor Initialization
    # This ensures the multimodal sub-module and its dependencies (timm, transformers, etc.) are intact
    try:
        test_predictor = MultiModalPredictor(label="dummy_label")
        print("MultiModalPredictor: Initialized successfully.")
    except Exception as e:
        print(f"MultiModalPredictor: Failed to initialize. Error: {e}")

    # 4. Check for Image-specific dependencies
    try:
        import timm
        print(f"TIMM (Image Models) Version: {timm.__version__}")
    except ImportError:
        print("ERROR: 'timm' library missing. This is required for image classification.")

if __name__ == "__main__":
    check_autogluon_multimodal()
